Chargement des données pour le traitement

In [ ]:
import pandas as pd
df = pd.read_csv(
'https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb'
)

In [ ]:
df.sample(20)

In [ ]:
df["candidat"] = df["prenom"] + " " + df["nom"]
df.sample(10)

## 2. Comparaison des scores départements aux moyennes nationales.

Q4. Créons un dataframe nommé score_departements stockant, pour chaque département, le nombre de vote obtenu pour chaque candidat et le score (en %).

In [ ]:
# Votes par candidat
score_departements = df.groupby(["code_departement", "candidat"]).agg(
    votes = ("voix", "sum")
).reset_index()

# Récupération des totaux sans tenir compte des votes adressé à personne
total_dep = df.loc[~df["candidat"].isna()].groupby(["code_departement"]).agg(
    total = ("voix", "sum")
).reset_index()

# Rajout du total
score_departements = score_departements.merge(right=total_dep, how="left", on="code_departement")
score_departements


# Calcul du score
score_departements["score"] = round(100 * score_departements["votes"] / score_departements["total"], 2)  # .astype(str)  + "%"
score_departements = score_departements.drop(columns="total")

In [ ]:
# affichage comme souhaité
x = score_departements.copy()
x["score"] = x["score"].astype(str)  + "%"
x

cellule de vérification ci-dessous avec le département 11

In [ ]:
x.loc[x["code_departement"] == "11"].sort_values("votes", ascending=False)

Q5. Refaissons le lien avec le niveau national pour comparer le score départemental avec le score national. Et nommons ce dataframe score_departements, nous allons le réutiliser par la suite.

In [ ]:
# Votes par candidat
nationale = df.groupby(["candidat"]).agg(
    votes_national = ("voix", "sum")
).reset_index()

# Récupération des totaux sans tenir compte des votes adressé à personne
total_nationale = sum(df.loc[~df["candidat"].isna()]["voix"])

# Calcul du score
nationale["score_national"] = round(100 * nationale["votes_national"] / total_nationale, 2) # .astype(str)  + "%"
nationale

# Jointure avec la table relatives aux départements
score_departements = score_departements.merge(right=nationale, on="candidat", how="left")
score_departements = score_departements.rename(columns={"votes": "votes_departement", "score": "score_departement"})
score_departements

In [ ]:
# Affichage comme demandé
x = score_departements.copy()
x["score_departement"] = x["score_departement"].astype(str)  + "%"
x["score_national"] = x["score_national"].astype(str)  + "%"
x

cellule de vérification ci-dessous avec le département 1

In [ ]:
x.loc[x["code_departement"] == "11"].sort_values("votes_departement", ascending=False)

Q6. Créons une variable surrepresentation qui compare, en relatif, les scores nationaux et départementaux

In [ ]:
score_departements["surrepresentation"] = round(((score_departements["score_departement"] / score_departements["score_national"]) - 1) * 100, 2)

In [ ]:
# Affichage comme demandé (avec le symbole %)
x = score_departements.copy()
x["score_departement"] = x["score_departement"].astype(str)  + "%"
x["score_national"] = x["score_national"].astype(str)  + "%"
x["surrepresentation"] = x["surrepresentation"].astype(str)  + "%"
x

Q7. Créons une fonction pour représenter une figure similaire à Figure 1 pour un candidat donné des
principales surreprésentations (en valeur absolue) par département.

In [ ]:
def display_surrepresentation(candidat: str, top: int=5):
    """
        Permet de représenter visuellement la surreprésentation des candidats

        Parameters
        ------------
            candidat : str
                Prenom et nom du candidat
            top : int
                le top a afficher (par défaut 5)

    """

    from matplotlib import pyplot as plt

    # Selection des données pour un candidat donné
    res = score_departements.loc[(
        (score_departements["candidat"] == candidat)
        )].sort_values(by="surrepresentation", key=abs, ascending=False).head(top) # On ordonne selon la valeur absolue de la surreprésentation

    res = res.sort_values("surrepresentation", ascending=True)

    plt.barh(y = res["code_departement"], width=res["surrepresentation"])
    plt.title("Top 5 des surreprésentation de " + candidat)
    plt.axvline(x=0)
    plt.show()

In [ ]:
display_surrepresentation(candidat="Éric ZEMMOUR")

In [ ]:
display_surrepresentation(candidat="Emmanuel MACRON", top=10)